# Fine-Tune LLM with Data Cache

This notebook demonstrates how to fine-tune a BERT model with Kubeflow TrainJob using data streaming from a distributed cache.

We will use an iceberg table with [the Amazon reviews dataset](https://huggingface.co/datasets/fancyzhx/amazon_polarity) loaded from data cache cluster, which provides efficient data streaming for distributed training workloads.

## Install Kubeflow SDK

In [1]:
# Install the Kubeflow SDK
!pip install -U kubeflow


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


## List Available Training Runtimes

Get available Kubeflow Training Runtimes using the list_runtimes() API.

In [2]:
from kubeflow.trainer import TrainerClient

client = TrainerClient()
for runtime in client.list_runtimes():
    print(f'{runtime}\n')

Runtime(name='deepspeed-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='deepspeed', image='ghcr.io/kubeflow/trainer/deepspeed-runtime:v2.3.0', num_nodes=1, device='Unknown', device_count='1'), kind=<RuntimeKind.CLUSTER_TRAINING_RUNTIME: 'ClusterTrainingRuntime'>, pretrained_model=None)

Runtime(name='jax-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='jax', image='nvcr.io/nvidia/jax:25.10-py3', num_nodes=1, device='Unknown', device_count='Unknown'), kind=<RuntimeKind.CLUSTER_TRAINING_RUNTIME: 'ClusterTrainingRuntime'>, pretrained_model=None)

Runtime(name='mlx-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='mlx', image='ghcr.io/kubeflow/trainer/mlx-runtime:v2.3.0', num_nodes=1, device='Unknown', device_count='1'), kind=<RuntimeKind.CLUSTER_TRAINING_RUNTIME: 'ClusterTrainingRuntime'>, pretrained_model=None)

R

## Define Training Function with DataCacheDataset

This training function uses DataCacheDataset to stream data from the cache cluster.

In [3]:
def train_func():
    import time
    import datetime
    from multiprocessing import current_process
    import torch.distributed as dist
    import pyarrow as pa
    import pyarrow.flight
    from pyarrow._flight import FlightClient
    import torch
    import os
    from torch.utils.data import IterableDataset
    from transformers import DistilBertForSequenceClassification, DistilBertTokenizer, AdamW

    
    # Define DataCacheDataset to access data from cache
    class DataCacheDataset(IterableDataset):
        def __init__(self, seed=0, batch_size=128):
            train_job_name = os.getenv("TRAIN_JOB_NAME")
            self.endpoint = f'grpc://{train_job_name}-cache-service:50051'
            self.seed = seed
            self.batch_size = batch_size

        def set_epoch(self, epoch):
            self.seed = self.seed + epoch

        def from_arrow_rb_to_tensor(self, chunk):
            return

        # Extract RecordBatches and split them into chunks
        def stream_recordbatch_chunks(self, recordbatch_stream, chunk_size):
            leftover = None
        
            for rb in recordbatch_stream:
                batch = rb.data
                if leftover is not None:
                    arrays = [
                        pa.concat_arrays([leftover.column(i), batch.column(i)])
                        for i in range(len(batch.columns))
                    ]
                    batch = pa.RecordBatch.from_arrays(arrays, batch.schema.names)

                for i in range(0, batch.num_rows, chunk_size):
                    chunk = batch.slice(i, chunk_size)
        
                    if chunk.num_rows < chunk_size:
                        leftover = chunk
                        break
                    else:
                        yield from self.from_arrow_rb_to_tensor(chunk)
                else:
                    leftover = None
        
            if leftover is not None and leftover.num_rows > 0:
                yield from self.from_arrow_rb_to_tensor(leftover)

        def __iter__(self):
            worker_info = torch.utils.data.get_worker_info()
            
            if worker_info is None:
                worker_id = 0
                num_workers = 1
            else:
                worker_id = worker_info.id
                num_workers = worker_info.num_workers

            world_size = dist.get_world_size()
            rank = dist.get_rank()
                
            # Calculate shard index
            total_workers = num_workers * world_size
            global_worker_id = worker_id * world_size + rank
            index = (global_worker_id + self.seed) % total_workers

            descriptor = pa.flight.FlightDescriptor.for_path(*[s for s in [str(index), str(total_workers)]])
            client = pa.flight.connect(self.endpoint)
            flight_info = client.get_flight_info(descriptor)

            for endpoint in flight_info.endpoints:
                for location in endpoint.locations:
                    _client = FlightClient(location.uri)
                    ticket = endpoint.ticket
                    reader = _client.do_get(ticket)
                    
                    yield from self.stream_recordbatch_chunks(reader, self.batch_size)

    # Define dataset for Amazon reviews
    class AmazonReviewDataset(DataCacheDataset):
        def __init__(self, tokenizer, seed=0):
            super().__init__(seed)
            self.tokenizer = tokenizer
            self.row_count = 0

        def from_arrow_rb_to_tensor(self, chunk):
            self.row_count = self.row_count + chunk.num_rows
            texts = chunk.column("content").to_pylist()
            labels = chunk.column("label").to_pylist()

            encoding = self.tokenizer(
                texts,
                padding="max_length",
                truncation=True,
                max_length=128,
                return_tensors="pt"
            )
        
            yield {
                "input_ids": encoding["input_ids"].squeeze(0),
                "attention_mask": encoding["attention_mask"].squeeze(0),
                "labels": torch.tensor(labels)
            }

    device, backend = ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    print(f"Using Device: {device}, Backend: {backend}")

    local_rank = int(os.getenv("LOCAL_RANK", 0))
    dist.init_process_group(backend=backend)
    print(
        "Distributed Training for WORLD_SIZE: {}, RANK: {}, LOCAL_RANK: {}".format(
            dist.get_world_size(),
            dist.get_rank(),
            local_rank,
        )
    )
    device = torch.device(f"{device}:{local_rank}")
    model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=5)
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

    start = time.time()
    dataset = AmazonReviewDataset(tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=None)

    # Training loop
    for epoch in range(1):
        dataset.set_epoch(epoch)
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            
            if dist.get_rank() == 0:
                print(f"Epoch {epoch}, Loss: {loss.item()}")

    dist.barrier()
    if dist.get_rank() == 0:
        print("Training is finished")

    end = time.time()
    print(f"Time taken: {end - start:.6f} seconds")
    print(f"Total rows processed: {dataset.row_count}")

    dist.destroy_process_group()

## Prepare Iceberg Dataset in SeaweedFS

In [5]:
import subprocess
import re

# 1. Clean up any previous runs
subprocess.run(["kubectl", "delete", "job", "iceberg-creator", "--ignore-not-found"], check=True)
subprocess.run(["kubectl", "delete", "configmap", "iceberg-script", "--ignore-not-found"], check=True)
subprocess.run(["kubectl", "delete", "trainjob", "--all", "--ignore-not-found"], check=True)

# 2. Write the Iceberg creation script
script_content = """
import pyarrow as pa
from pyiceberg.catalog import load_catalog

catalog = load_catalog(
    "seaweedfs",
    **{
        "type": "sql",
        "uri": "sqlite:////tmp/catalog.db",
        "s3.endpoint": "http://seaweedfs-s3.seaweedfs.svc.cluster.local:8333",
        "s3.access-key-id": "dummy",
        "s3.secret-access-key": "dummy",
        "s3.region": "us-east-1",
    },
)

schema = pa.schema([
    ("content", pa.string()),
    ("label", pa.int64()),
])

catalog.create_namespace_if_not_exists("default")
table = catalog.create_table_if_not_exists(
    "default.amazon_reviews",
    schema=schema,
    location="s3://mlpipeline/amazon_reviews",
)

df = pa.Table.from_arrays(
    [
        pa.array([
            "Great product, works perfectly!",
            "Terrible quality, broken on arrival.",
            "Average experience, fast shipping."
        ]),
        pa.array([4, 0, 2]),
    ],
    schema=schema,
)
table.append(df)

print("--- METADATA_PATH_START ---")
print(table.metadata_location)
print("--- METADATA_PATH_END ---")
"""

with open("/tmp/create_iceberg.py", "w") as f:
    f.write(script_content)

# 3. Create ConfigMap from the script
subprocess.run(["kubectl", "create", "configmap", "iceberg-script", "--from-file=/tmp/create_iceberg.py"], check=True)

# 4. Deploy the Job with sql-sqlite support included
job_manifest = """
apiVersion: batch/v1
kind: Job
metadata:
  name: iceberg-creator
spec:
  ttlSecondsAfterFinished: 300
  template:
    spec:
      restartPolicy: Never
      containers:
      - name: creator
        image: python:3.11-slim
        command:
        - "sh"
        - "-c"
        - "pip install --quiet 'pyiceberg[s3fs,pyarrow,sql-sqlite]' pyarrow && python3 /scripts/create_iceberg.py"
        volumeMounts:
        - name: script-vol
          mountPath: /scripts
      volumes:
      - name: script-vol
        configMap:
          name: iceberg-script
"""

subprocess.run(["kubectl", "apply", "-f", "-"], input=job_manifest, text=True, check=True)

print("Job submitted. Installing dependencies and creating Iceberg table...")

# 5. Wait for the Job to finish (up to 5 minutes)
subprocess.run(
    ["kubectl", "wait", "--for=condition=complete", "--timeout=600s", "job/iceberg-creator"],
    check=True
)

# 6. Fetch logs and extract the generated S3 path
logs_res = subprocess.run(["kubectl", "logs", "-l", "job-name=iceberg-creator"], capture_output=True, text=True, check=True)
output = logs_res.stdout

match = re.search(r"--- METADATA_PATH_START ---\s+(s3://\S+)\s+--- METADATA_PATH_END ---", output)
if match:
    GENERATED_METADATA_LOC = match.group(1).strip()
    print(f"\nSuccessfully created Iceberg Table!\nMetadata Location: {GENERATED_METADATA_LOC}")
else:
    print("Execution output:\n", output)
    raise RuntimeError("Failed to extract metadata location.")

job.batch "iceberg-creator" deleted from default namespace
configmap "iceberg-script" deleted from default namespace
No resources found
configmap/iceberg-script created
job.batch/iceberg-creator created
Job submitted. Installing dependencies and creating Iceberg table...
job.batch/iceberg-creator condition met

Successfully created Iceberg Table!
Metadata Location: s3://mlpipeline/amazon_reviews/metadata/00001-0449bdd8-6fae-46c4-828d-7c5fef683f88.metadata.json


## Create TrainJob with Data Cache

Use the train() API with DataCacheInitializer to create a TrainJob that streams data from cache.

In [6]:
from kubeflow.trainer import CustomTrainer, DataCacheInitializer, Initializer

# Get the cache runtime
torch_cache_runtime = None
for runtime in client.list_runtimes():
    if runtime.name == "torch-distributed-with-cache":
        torch_cache_runtime = runtime
        break

if torch_cache_runtime is None:
    raise ValueError("torch-distributed-with-cache runtime not found. Please ensure the runtime is installed.")

# Configure your data cache settings
# Replace these values with your actual metadata location and storage URI
METADATA_LOC = GENERATED_METADATA_LOC
STORAGE_URI = "cache://default/amazon_reviews"
IAM_ROLE= "dummy"

job_name = client.train(
    trainer=CustomTrainer(
        func=train_func,
        # Set how many PyTorch nodes you want to use for distributed training
        num_nodes=1,
        # Set the resources for each PyTorch node
        resources_per_node={
            "cpu": 1,
            "memory": "4Gi",
            # Uncomment to use GPUs
            #"nvidia.com/gpu": 1,
        },
        packages_to_install=["pyarrow==19.0.0", "transformers==4.45.2"]
    ),
    # Dataset config with data cache initializer
    initializer=Initializer(
        dataset=DataCacheInitializer(
            num_data_nodes=2,
            metadata_loc=METADATA_LOC,
            storage_uri=STORAGE_URI,
            iam_role=IAM_ROLE
        )
    ),
    runtime=torch_cache_runtime,
)

print(f"TrainJob created: {job_name}")

TrainJob created: p292317f52b1


## Check the TrainJob Info

Use the list_jobs() and get_job() APIs to get information about the created TrainJob.

In [7]:
for job in client.list_jobs():
    print(f"TrainJob: {job.name}, Status: {job.status}, Created at: {job.creation_timestamp}")

TrainJob: p292317f52b1, Status: Created, Created at: 2026-11-10 16:11:52+00:00


In [8]:
# Get details about the TrainJob steps
for step in client.get_job(name=job_name).steps:
    print(f"Step: {step.name}, Status: {step.status}, Devices: {step.device} x {step.device_count}\n")

Step: dataset-initializer, Status: Running, Devices: Unknown x Unknown



## Watch the TrainJob Logs

Use the get_job_logs() API to follow the TrainJob logs in real-time.

In [9]:
# Wait for the job to reach running state
client.wait_for_job_status(job_name, status={"Running"})
print("Job is in running state")

# Follow the logs
for line in client.get_job_logs(job_name, follow=True):
    print(line)
    

Job is in running state
Using Device: cpu, Backend: gloo
Distributed Training for WORLD_SIZE: 1, RANK: 0, LOCAL_RANK: 0
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/root/.local/lib/python3.12/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Epoch 0, Loss: 1.618381142616272
Training is finished
Time taken: 16.687102 seconds
Total rows processed: 3


## Verify TrainJob Completion

Check the TrainJob status to ensure it completed successfully.

In [10]:
client.wait_for_job_status(name=job_name, timeout=300)

TrainJob(name='p292317f52b1', runtime=Runtime(name='torch-distributed-with-cache', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='pytorch/pytorch:2.13.0-cuda13.0-cudnn9-runtime', num_nodes=1, device='Unknown', device_count='Unknown'), kind=<RuntimeKind.CLUSTER_TRAINING_RUNTIME: 'ClusterTrainingRuntime'>, pretrained_model=None), steps=[Step(name='dataset-initializer', status='Succeeded', pod_name='p292317f52b1-dataset-initializer-0-0-jp4hl', device='Unknown', device_count='Unknown'), Step(name='node-0', status='Succeeded', pod_name='p292317f52b1-node-0-0-4jjmt', device='cpu', device_count='1')], num_nodes=1, creation_timestamp=datetime.datetime(2026, 11, 10, 16, 11, 52, tzinfo=TzInfo(0)), status='Complete')

## Delete the TrainJob

When the TrainJob is finished, you can delete the resource.

In [13]:
# Uncomment to delete the job
client.delete_job(job_name)